In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Carregar os dados
df = pd.read_csv(r'C:\Users\marya\OneDrive\Área de Trabalho\PastHires\phf_play_by_play.csv')

# Filtrar apenas eventos de finalização (Shots e Goals)
eventos_chute = ['Shot', 'Goal', 'PP Goal', 'Short Handed Goal']
df_model = df[df['event'].isin(eventos_chute)].copy()

# Se o evento contém "Goal", é 1, caso contrário (apenas Shot), é 0
df_model['is_goal'] = df_model['event'].str.contains('Goal').astype(int)

# transformar variáveis categóricas em números
df_model['situation_numeric'] = df_model['on_ice_situation'].factorize()[0]
df_model['period_id'] = df_model['period_id'].astype(int)

# Seleciona as colunas para o modelo
features = ['period_id', 'home_goals', 'away_goals', 'situation_numeric', 'sec_from_start']
X = df_model[features]
y = df_model['is_goal']

# Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modelo
modelo_nhl = RandomForestClassifier(n_estimators=100)
modelo_nhl.fit(X_train, y_train)

# Resultado
y_pred = modelo_nhl.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.92      0.90       998
           1       0.23      0.16      0.19       154

    accuracy                           0.82      1152
   macro avg       0.55      0.54      0.54      1152
weighted avg       0.79      0.82      0.80      1152



In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# limpeza e controle de Dados
df = pd.read_csv(r'C:\Users\marya\OneDrive\Área de Trabalho\PastHires\phf_play_by_play.csv')

# Criar uma coluna alvo: 1 se houve penalidade, 0 se não
df['has_penalty'] = df['penalty'].fillna(0).apply(lambda x: 1 if x != 0 else 0)

# Transformar variáveis de texto em números
le = LabelEncoder()
df['team_encoded'] = le.fit_transform(df['team'].astype(str))
df['situation_encoded'] = le.fit_transform(df['on_ice_situation'].astype(str))

features = ['sec_from_start', 'home_goals', 'away_goals', 'team_encoded', 'situation_encoded']
X = df[features]
y = df['has_penalty']

# Treino do Modelo
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf = RandomForestClassifier(n_estimators=100, class_weight='balanced')
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=features)
print("mais causas de penalidades:")
print(importances.sort_values(ascending=False))

O que mais causa penalidades segundo o modelo:
situation_encoded    0.581288
sec_from_start       0.302953
team_encoded         0.041660
home_goals           0.037396
away_goals           0.036703
dtype: float64
